In [5]:
%pip install psycopg[binary]

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import psycopg

In [7]:
from psycopg import sql

In [8]:
%pip install random

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement random (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for random


In [9]:
import random
import string
import pandas as pd


def generate_site_code():
    letters = "".join(random.choices(string.ascii_uppercase, k=3))
    numbers = "".join(random.choices(string.digits, k=3))
    return letters + numbers


def generate_site_data(count):
    data = []
    used_site_codes = set()
    used_coordinates = set()

    while len(data) < count:

        site_code = generate_site_code()
        latitude = round(random.uniform(-90, 90), 2)
        longitude = round(random.uniform(-180, 180), 2)

        # Skip duplicate site codes
        if site_code in used_site_codes:
            continue

        # Skip duplicate coordinates
        if (latitude, longitude) in used_coordinates:
            continue

        used_site_codes.add(site_code)
        used_coordinates.add((latitude, longitude))

        data.append(
            {
                "site_code": site_code,
                "latitude": latitude,
                "longitude": longitude,
            }
        )

    return pd.DataFrame(data)

In [10]:
def create_db_meta(db_name):
    try:
        with psycopg.connect(
            dbname="csv_database",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
            autocommit=True,
        ) as conn:
            with conn.cursor() as cur:
                query = sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_name))
                cur.execute(query)
                print(f"Database '{db_name}' created successfully!")

    except psycopg.Error as e:
        print(f"An error occurred: {e}")

In [11]:
create_db_meta("meta")

Database 'meta' created successfully!


In [13]:
def create_table_meta():
    try:
        with psycopg.connect(
            dbname="meta",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
        ) as conn:

            with conn.cursor() as cur:
                cur.execute("""
                    CREATE TABLE IF NOT EXISTS metadata (
                        site_name VARCHAR(100) Not NULL,
                        latitude DOUBLE PRECISION Not NULL,
                        longitude DOUBLE PRECISION Not NULL
                    );
                """)

            conn.commit()
            print("metadata table created.")

    except psycopg.Error as e:
        print(e)

In [14]:
create_table_meta()

metadata table created.


In [15]:
def insert_sites():

    sites_df = generate_site_data(10000)

    with psycopg.connect(
        dbname="meta", user="postgres", password="123789", host="localhost", port="5000"
    ) as conn:
        with conn.cursor() as cur:
            for _, row in sites_df.iterrows():
                cur.execute(
                    """
                    INSERT INTO metadata
                    (site_name, latitude, longitude)
                    VALUES (%s, %s, %s)                                                                         
                    """,
                    (row["site_code"], row["latitude"], row["longitude"]),
                )
        conn.commit()

    print("Sites inserted successfully!")

In [16]:
insert_sites()

Sites inserted successfully!


In [17]:
def get_sites():

    with psycopg.connect(
        dbname="meta", user="postgres", password="123789", host="localhost", port="5000"
    ) as conn:

        with conn.cursor() as cur:
            cur.execute(""" 
                SELECT site_name, latitude, longitude
                FROM metadata
            """)

            sites = cur.fetchall()

    return sites

In [18]:
data = list(get_sites())

In [19]:
type(data[0])

tuple

In [20]:
len(data)

10000

In [21]:
%pip install openmeteo-requests
%pip install requests-cache retry-requests numpy pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
def create_db_site_weather(db_name):
    try:
        with psycopg.connect(
            dbname="csv_database",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
            autocommit=True,
        ) as conn:
            with conn.cursor() as cur:
                query = sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_name))
                cur.execute(query)
                print(f"Database '{db_name}' created successfully!")

    except psycopg.Error as e:
        print(f"An error occurred: {e}")

In [24]:
create_db_site_weather("site_weather")

Database 'site_weather' created successfully!


In [ ]:
import time
import datetime
import requests
import psycopg

# ============================================================
# SETTINGS
# ============================================================

DB_USER = "postgres"
DB_PASSWORD = "123789"
DB_HOST = "localhost"
DB_PORT = "5000"

OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"

START_DATE = "2026-08-11"
END_DATE = "2026-08-11"

REQUEST_TIMEOUT = 120

# Normal delay between successful requests
REQUEST_DELAY = 0.2

# Fallback waits
MINUTE_WAIT = 60
HOUR_WAIT = 3600

# Maximum exponential backoff for temporary errors
MAX_BACKOFF = 300


# ============================================================
# CREATE TABLE IF NOT EXISTS
# ============================================================

with psycopg.connect(
    dbname="site_weather",
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            CREATE TABLE IF NOT EXISTS site_weather (

                site_name VARCHAR(100) NOT NULL,

                time_interval TIMESTAMPTZ NOT NULL,

                temperature REAL NOT NULL,

                humidity REAL NOT NULL,

                solar_radiance REAL NOT NULL,

                UNIQUE (
                    site_name,
                    time_interval
                )
            );
            """)

    conn.commit()


# ============================================================
# RETRY-AFTER
# ============================================================


def get_retry_after(response):

    value = response.headers.get("Retry-After")

    if value is None:
        return None

    # Retry-After can be seconds
    try:

        seconds = float(value)

        if seconds >= 0:
            return int(seconds)

    except (ValueError, TypeError):

        pass

    # Retry-After can also be an HTTP date
    try:

        from email.utils import parsedate_to_datetime

        retry_date = parsedate_to_datetime(value)

        if retry_date.tzinfo is None:

            retry_date = retry_date.replace(tzinfo=datetime.timezone.utc)

        now = datetime.datetime.now(datetime.timezone.utc)

        seconds = (retry_date - now).total_seconds()

        return max(0, int(seconds))

    except Exception:

        return None


# ============================================================
# DETECT LIMIT TYPE
# ============================================================


def detect_rate_limit_type(response):

    text = response.text.lower()

    headers = {str(k).lower(): str(v).lower() for k, v in response.headers.items()}

    header_text = " ".join(headers.values())

    combined_text = text + " " + header_text

    # --------------------------------------------------------
    # Explicit hourly indicators
    # --------------------------------------------------------

    hourly_words = [
        "hour",
        "hourly",
        "per hour",
        "hour limit",
        "hourly limit",
        "hourly quota",
        "quota per hour",
    ]

    for word in hourly_words:

        if word in combined_text:

            return "hour"

    # --------------------------------------------------------
    # Explicit minute indicators
    # --------------------------------------------------------

    minute_words = [
        "minute",
        "per minute",
        "minute limit",
        "minute quota",
        "requests/min",
    ]

    for word in minute_words:

        if word in combined_text:

            return "minute"

    return None


# ============================================================
# CALCULATE RATE-LIMIT WAIT
# ============================================================


def get_rate_limit_wait(response):

    # --------------------------------------------------------
    # 1. Retry-After has highest priority
    # --------------------------------------------------------

    retry_after = get_retry_after(response)

    if retry_after is not None:

        return retry_after, "Retry-After"

    # --------------------------------------------------------
    # 2. Detect minute/hour limit
    # --------------------------------------------------------

    limit_type = detect_rate_limit_type(response)

    if limit_type == "hour":

        return HOUR_WAIT, "hourly limit"

    if limit_type == "minute":

        return MINUTE_WAIT, "minute limit"

    # --------------------------------------------------------
    # 3. Unknown 429
    #
    # IMPORTANT:
    # Don't keep hitting API every 60 seconds.
    #
    # Safest fallback = one hour.
    # --------------------------------------------------------

    return HOUR_WAIT, "unknown 429 - safe hourly wait"


# ============================================================
# FETCH WEATHER FOR ONE SITE
# ============================================================


def fetch_site_weather(site_name, latitude, longitude):

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": ["temperature_2m", "relative_humidity_2m", "direct_radiation"],
        "start_date": START_DATE,
        "end_date": END_DATE,
    }

    temporary_retry_count = 0

    while True:

        try:

            print(f"\nAPI request -> {site_name}")

            response = requests.get(
                OPEN_METEO_URL, params=params, timeout=REQUEST_TIMEOUT
            )

            status = response.status_code

            # =================================================
            # SUCCESS
            # =================================================

            if status == 200:

                result = response.json()

                if "hourly" not in result:

                    raise RuntimeError(
                        f"API returned no hourly data " f"for {site_name}"
                    )

                hourly = result["hourly"]

                times = hourly.get("time", [])

                temperatures = hourly.get("temperature_2m", [])

                humidity = hourly.get("relative_humidity_2m", [])

                radiation = hourly.get("direct_radiation", [])

                # ------------------------------------------------
                # Validate response
                # ------------------------------------------------

                if not (
                    len(times) == 24
                    and len(temperatures) == 24
                    and len(humidity) == 24
                    and len(radiation) == 24
                ):

                    raise RuntimeError(
                        f"Incomplete API response "
                        f"for {site_name}: "
                        f"time={len(times)}, "
                        f"temperature={len(temperatures)}, "
                        f"humidity={len(humidity)}, "
                        f"radiation={len(radiation)}"
                    )

                print(f"API SUCCESS -> {site_name}")

                return (times, temperatures, humidity, radiation)

            # =================================================
            # 429 RATE LIMIT
            # =================================================

            elif status == 429:

                wait_seconds, reason = get_rate_limit_wait(response)

                print("\n" + "=" * 70)

                print("429 RATE LIMIT")

                print(f"Site: {site_name}")

                print(f"Reason: {reason}")

                print(f"Retry-After: " f"{response.headers.get('Retry-After')}")

                print(f"Waiting: " f"{wait_seconds} seconds")

                print("=" * 70)

                # ------------------------------------------------
                # WAIT
                # ------------------------------------------------

                time.sleep(wait_seconds)

                print(f"\nRetrying SAME site: " f"{site_name}")

                # Reset temporary errors
                temporary_retry_count = 0

                continue

            # =================================================
            # 408 REQUEST TIMEOUT
            # =================================================

            elif status == 408:

                temporary_retry_count += 1

                wait_seconds = min(2**temporary_retry_count, MAX_BACKOFF)

                print(f"HTTP 408 for {site_name}")

                print(f"Retrying in " f"{wait_seconds} seconds...")

                time.sleep(wait_seconds)

                continue

            # =================================================
            # 5XX SERVER ERRORS
            # =================================================

            elif 500 <= status <= 599:

                temporary_retry_count += 1

                wait_seconds = min(2**temporary_retry_count, MAX_BACKOFF)

                print(f"HTTP {status} " f"for {site_name}")

                print(f"Retrying in " f"{wait_seconds} seconds...")

                time.sleep(wait_seconds)

                continue

            # =================================================
            # OTHER 4XX
            # =================================================

            elif 400 <= status <= 499:

                raise RuntimeError(
                    f"Permanent API error "
                    f"for {site_name}: "
                    f"HTTP {status}\n"
                    f"{response.text[:1000]}"
                )

            # =================================================
            # UNKNOWN STATUS
            # =================================================

            else:

                temporary_retry_count += 1

                wait_seconds = min(2**temporary_retry_count, MAX_BACKOFF)

                print(f"Unexpected HTTP " f"{status} for {site_name}")

                print(f"Retrying in " f"{wait_seconds} seconds...")

                time.sleep(wait_seconds)

        # =====================================================
        # NETWORK TIMEOUT
        # =====================================================

        except requests.Timeout as e:

            temporary_retry_count += 1

            wait_seconds = min(2**temporary_retry_count, MAX_BACKOFF)

            print(f"Network timeout " f"for {site_name}: {e}")

            print(f"Retrying in " f"{wait_seconds} seconds...")

            time.sleep(wait_seconds)

        # =====================================================
        # CONNECTION ERROR
        # =====================================================

        except requests.ConnectionError as e:

            temporary_retry_count += 1

            wait_seconds = min(2**temporary_retry_count, MAX_BACKOFF)

            print(f"Connection error " f"for {site_name}: {e}")

            print(f"Retrying in " f"{wait_seconds} seconds...")

            time.sleep(wait_seconds)

        # =====================================================
        # OTHER REQUEST ERROR
        # =====================================================

        except requests.RequestException as e:

            temporary_retry_count += 1

            wait_seconds = min(2**temporary_retry_count, MAX_BACKOFF)

            print(f"Request error " f"for {site_name}: {e}")

            print(f"Retrying in " f"{wait_seconds} seconds...")

            time.sleep(wait_seconds)


# ============================================================
# SAVE WEATHER DATA
# ============================================================


def save_site_weather(cursor, site_name, times, temperatures, humidity, radiation):

    inserted = 0

    for i in range(24):

        timestamp = datetime.datetime.fromisoformat(times[i]).replace(
            tzinfo=datetime.timezone.utc
        )

        cursor.execute(
            """
            INSERT INTO site_weather (
                site_name,
                time_interval,
                temperature,
                humidity,
                solar_radiance
            )
            VALUES (
                %s,
                %s,
                %s,
                %s,
                %s
            )
            ON CONFLICT (
                site_name,
                time_interval
            )
            DO NOTHING;
            """,
            (
                site_name,
                timestamp,
                float(temperatures[i]),
                float(humidity[i]),
                float(radiation[i]),
            ),
        )

        if cursor.rowcount == 1:

            inserted += 1

    return inserted


# ============================================================
# PROCESS ALL SITES
# ============================================================

processed = 0
total_inserted = 0


with psycopg.connect(
    dbname="site_weather",
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        for index, row in enumerate(data, start=1):

            site_name = row[0]

            latitude = float(row[1])

            longitude = float(row[2])

            # =================================================
            # IMPORTANT:
            # CHECK WHETHER THIS SITE IS ALREADY COMPLETE
            # =================================================

            cur.execute(
                """
                SELECT COUNT(*)
                FROM site_weather
                WHERE site_name = %s;
                """,
                (site_name,),
            )

            existing_rows = cur.fetchone()[0]

            if existing_rows == 24:

                print(
                    f"\nSKIPPING "
                    f"{index}/{len(data)} "
                    f"{site_name} "
                    f"(already has 24 rows)"
                )

                processed += 1

                continue

            # =================================================
            # PROCESS SITE
            # =================================================

            print("\n" + "=" * 70)

            print(f"PROCESSING " f"{index}/{len(data)}")

            print(f"Site: {site_name}")

            print(f"Latitude: {latitude}")

            print(f"Longitude: {longitude}")

            print("=" * 70)

            # -------------------------------------------------
            # API
            # -------------------------------------------------

            times, temperatures, humidity, radiation = fetch_site_weather(
                site_name, latitude, longitude
            )

            # -------------------------------------------------
            # DATABASE
            # -------------------------------------------------

            inserted = save_site_weather(
                cur, site_name, times, temperatures, humidity, radiation
            )

            # -------------------------------------------------
            # COMMIT IMMEDIATELY
            # -------------------------------------------------

            conn.commit()

            processed += 1

            total_inserted += inserted

            print(f"\nSUCCESS -> {site_name}")

            print(f"Inserted: {inserted} rows")

            print(f"Progress: " f"{processed}/{len(data)}")

            print(f"Total inserted this run: " f"{total_inserted}")

            # -------------------------------------------------
            # NORMAL DELAY
            # -------------------------------------------------

            time.sleep(REQUEST_DELAY)


# ============================================================
# FINAL CHECK
# ============================================================

with psycopg.connect(
    dbname="site_weather",
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT COUNT(DISTINCT site_name)
            FROM site_weather;
            """)

        distinct_sites = cur.fetchone()[0]

        cur.execute("""
            SELECT COUNT(*)
            FROM site_weather;
            """)

        total_rows = cur.fetchone()[0]


print("\n" + "=" * 70)
print("FINAL RESULT")
print("=" * 70)

print(f"Expected sites : {len(data)}")

print(f"Actual sites   : {distinct_sites}")

print(f"Expected rows  : {len(data) * 24}")

print(f"Actual rows    : {total_rows}")

print("=" * 70)


if distinct_sites == len(data) and total_rows == len(data) * 24:

    print("SUCCESS: " "10,000 sites × 24 = 240,000 rows.")

else:

    print("Pipeline is not complete yet.")


PROCESSING 1/10000
Site: AZJ088
Latitude: -78.8
Longitude: -26.05

API request -> AZJ088
API SUCCESS -> AZJ088

SUCCESS -> AZJ088
Inserted: 24 rows
Progress: 1/10000
Total inserted this run: 24

PROCESSING 2/10000
Site: PKL930
Latitude: -72.14
Longitude: 167.78

API request -> PKL930
API SUCCESS -> PKL930

SUCCESS -> PKL930
Inserted: 24 rows
Progress: 2/10000
Total inserted this run: 48

PROCESSING 3/10000
Site: TNJ749
Latitude: 0.04
Longitude: 72.46

API request -> TNJ749
API SUCCESS -> TNJ749

SUCCESS -> TNJ749
Inserted: 24 rows
Progress: 3/10000
Total inserted this run: 72

PROCESSING 4/10000
Site: ZUU807
Latitude: -3.06
Longitude: -34.5

API request -> ZUU807
API SUCCESS -> ZUU807

SUCCESS -> ZUU807
Inserted: 24 rows
Progress: 4/10000
Total inserted this run: 96

PROCESSING 5/10000
Site: KRM656
Latitude: 70.05
Longitude: -46.09

API request -> KRM656
API SUCCESS -> KRM656

SUCCESS -> KRM656
Inserted: 24 rows
Progress: 5/10000
Total inserted this run: 120

PROCESSING 6/10000
Site: 

In [24]:
# ============================================================
# CODE STARTS AFTER:
#
# data = list(get_sites())
# ============================================================

import time
import datetime
import requests
import psycopg

# ============================================================
# SETTINGS
# ============================================================

DB_USER = "postgres"
DB_PASSWORD = "123789"
DB_HOST = "localhost"
DB_PORT = "5000"

OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"

START_DATE = "2026-08-11"
END_DATE = "2026-08-11"

# Small delay between successful API requests
REQUEST_DELAY = 0.2

# Fallback waits when API does NOT provide Retry-After
MINUTE_LIMIT_WAIT = 60
HOURLY_LIMIT_WAIT = 3600

# Maximum wait for temporary network/server errors
MAX_BACKOFF = 300

# HTTP timeout
REQUEST_TIMEOUT = 120


# ============================================================
# CREATE DESTINATION TABLE
# ============================================================

with psycopg.connect(
    dbname="site_weather",
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            CREATE TABLE IF NOT EXISTS site_weather (

                site_name VARCHAR(100) NOT NULL,

                time_interval TIMESTAMPTZ NOT NULL,

                temperature REAL NOT NULL,

                humidity REAL NOT NULL,

                solar_radiance REAL NOT NULL,

                UNIQUE (
                    site_name,
                    time_interval
                )
            );
            """)

    conn.commit()


print(f"Total sites received from meta: {len(data)}")


# ============================================================
# RETRY-AFTER PARSER
# ============================================================


def get_retry_after(response):

    value = response.headers.get("Retry-After")

    if value is None:
        return None

    # --------------------------------------------------------
    # Retry-After can be seconds
    # --------------------------------------------------------

    try:

        seconds = float(value)

        if seconds >= 0:
            return int(seconds)

    except (ValueError, TypeError):

        pass

    # --------------------------------------------------------
    # Retry-After can also be an HTTP date
    # --------------------------------------------------------

    try:

        from email.utils import parsedate_to_datetime

        retry_date = parsedate_to_datetime(value)

        if retry_date.tzinfo is None:

            retry_date = retry_date.replace(tzinfo=datetime.timezone.utc)

        now = datetime.datetime.now(datetime.timezone.utc)

        seconds = (retry_date - now).total_seconds()

        return max(0, int(seconds))

    except Exception:

        return None


# ============================================================
# DETERMINE RATE LIMIT WAIT
# ============================================================


def get_rate_limit_wait(response):

    # Retry-After always gets first priority
    retry_after = get_retry_after(response)

    if retry_after is not None:

        return retry_after

    # --------------------------------------------------------
    # No Retry-After
    # Determine minute/hour from response
    # --------------------------------------------------------

    text = response.text.lower()

    if "hourly" in text or "hour" in text:

        return HOURLY_LIMIT_WAIT

    return MINUTE_LIMIT_WAIT


# ============================================================
# FETCH ONE SITE
# ============================================================


def fetch_site_weather(site_name, latitude, longitude):

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": ["temperature_2m", "relative_humidity_2m", "direct_radiation"],
        "start_date": START_DATE,
        "end_date": END_DATE,
    }

    retry_count = 0

    while True:

        try:

            print(f"Fetching: {site_name}")

            # ------------------------------------------------
            # Direct request
            # NO SESSION
            # NO CACHE
            # ------------------------------------------------

            response = requests.get(
                OPEN_METEO_URL, params=params, timeout=REQUEST_TIMEOUT
            )

            status = response.status_code

            # =================================================
            # SUCCESS
            # =================================================

            if status == 200:

                result = response.json()

                if "hourly" not in result:

                    raise RuntimeError(f"No hourly data returned " f"for {site_name}")

                hourly = result["hourly"]

                times = hourly.get("time", [])

                temperatures = hourly.get("temperature_2m", [])

                humidity = hourly.get("relative_humidity_2m", [])

                radiation = hourly.get("direct_radiation", [])

                # ------------------------------------------------
                # Validate returned data
                # ------------------------------------------------

                if not (
                    len(times) == 24
                    and len(temperatures) == 24
                    and len(humidity) == 24
                    and len(radiation) == 24
                ):

                    raise RuntimeError(
                        f"Invalid hourly data for "
                        f"{site_name}: "
                        f"times={len(times)}, "
                        f"temperature={len(temperatures)}, "
                        f"humidity={len(humidity)}, "
                        f"radiation={len(radiation)}"
                    )

                return (times, temperatures, humidity, radiation)

            # =================================================
            # 429 RATE LIMIT
            # =================================================

            if status == 429:

                wait_seconds = get_rate_limit_wait(response)

                print("\n" + "=" * 70)

                print("API RATE LIMIT REACHED")

                print(f"Site: {site_name}")

                print(f"HTTP 429")

                print(f"Retry-After: " f"{response.headers.get('Retry-After')}")

                print(f"Waiting {wait_seconds} seconds...")

                print("=" * 70)

                # ------------------------------------------------
                # Wait before retrying SAME SITE
                # ------------------------------------------------

                time.sleep(wait_seconds)

                retry_count = 0

                continue

            # =================================================
            # 408 TIMEOUT
            # =================================================

            if status == 408:

                retry_count += 1

                wait_seconds = min(2**retry_count, MAX_BACKOFF)

                print(f"HTTP 408 for {site_name}. " f"Retrying in {wait_seconds}s...")

                time.sleep(wait_seconds)

                continue

            # =================================================
            # 5XX SERVER ERROR
            # =================================================

            if 500 <= status <= 599:

                retry_count += 1

                wait_seconds = min(2**retry_count, MAX_BACKOFF)

                print(
                    f"HTTP {status} for {site_name}. " f"Retrying in {wait_seconds}s..."
                )

                time.sleep(wait_seconds)

                continue

            # =================================================
            # OTHER 4XX
            # =================================================

            if 400 <= status <= 499:

                raise RuntimeError(
                    f"Permanent API error for "
                    f"{site_name}: "
                    f"HTTP {status} - "
                    f"{response.text[:500]}"
                )

            # =================================================
            # UNEXPECTED STATUS
            # =================================================

            retry_count += 1

            wait_seconds = min(2**retry_count, MAX_BACKOFF)

            print(
                f"Unexpected HTTP {status} "
                f"for {site_name}. "
                f"Retrying in {wait_seconds}s..."
            )

            time.sleep(wait_seconds)

        # =====================================================
        # NETWORK TIMEOUT
        # =====================================================

        except requests.Timeout as e:

            retry_count += 1

            wait_seconds = min(2**retry_count, MAX_BACKOFF)

            print(f"Network timeout for " f"{site_name}: {e}")

            print(f"Retrying in " f"{wait_seconds}s...")

            time.sleep(wait_seconds)

        # =====================================================
        # CONNECTION ERROR
        # =====================================================

        except requests.ConnectionError as e:

            retry_count += 1

            wait_seconds = min(2**retry_count, MAX_BACKOFF)

            print(f"Connection error for " f"{site_name}: {e}")

            print(f"Retrying in " f"{wait_seconds}s...")

            time.sleep(wait_seconds)

        # =====================================================
        # OTHER REQUEST ERRORS
        # =====================================================

        except requests.RequestException as e:

            retry_count += 1

            wait_seconds = min(2**retry_count, MAX_BACKOFF)

            print(f"Request error for " f"{site_name}: {e}")

            print(f"Retrying in " f"{wait_seconds}s...")

            time.sleep(wait_seconds)


# ============================================================
# SAVE ONE SITE
# ============================================================


def save_site_weather(cursor, site_name, times, temperatures, humidity, radiation):

    inserted = 0

    for i in range(24):

        time_interval = datetime.datetime.fromisoformat(times[i]).replace(
            tzinfo=datetime.timezone.utc
        )

        cursor.execute(
            """
            INSERT INTO site_weather (
                site_name,
                time_interval,
                temperature,
                humidity,
                solar_radiance
            )
            VALUES (
                %s,
                %s,
                %s,
                %s,
                %s
            )
            ON CONFLICT (
                site_name,
                time_interval
            )
            DO NOTHING;
            """,
            (
                site_name,
                time_interval,
                float(temperatures[i]),
                float(humidity[i]),
                float(radiation[i]),
            ),
        )

        if cursor.rowcount == 1:

            inserted += 1

    return inserted


# ============================================================
# PROCESS ALL SITES
# ============================================================

processed = 0
total_inserted = 0


with psycopg.connect(
    dbname="site_weather",
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        for index, row in enumerate(data, start=1):

            site_name = row[0]

            latitude = float(row[1])

            longitude = float(row[2])

            print("\n" + "=" * 70)

            print(f"PROGRESS: " f"{index}/{len(data)}")

            print(f"SITE: {site_name}")

            print(f"LATITUDE: {latitude}")

            print(f"LONGITUDE: {longitude}")

            print("=" * 70)

            # =================================================
            # API CALL
            # =================================================

            times, temperatures, humidity, radiation = fetch_site_weather(
                site_name, latitude, longitude
            )

            # =================================================
            # SAVE API RESULT
            # =================================================

            inserted = save_site_weather(
                cur, site_name, times, temperatures, humidity, radiation
            )

            # =================================================
            # COMMIT IMMEDIATELY
            # =================================================

            conn.commit()

            processed += 1

            total_inserted += inserted

            print(f"SUCCESS: {site_name}")

            print(f"Rows saved: {inserted}/24")

            print(f"Progress: " f"{processed}/{len(data)}")

            print(f"Total rows saved: " f"{total_inserted}")

            # =================================================
            # NORMAL DELAY
            # =================================================

            time.sleep(REQUEST_DELAY)


# ============================================================
# FINAL VERIFICATION
# ============================================================

with psycopg.connect(
    dbname="site_weather",
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT COUNT(DISTINCT site_name)
            FROM site_weather;
            """)

        distinct_sites = cur.fetchone()[0]

        cur.execute("""
            SELECT COUNT(*)
            FROM site_weather;
            """)

        total_rows = cur.fetchone()[0]


print("\n" + "=" * 70)
print("FINAL RESULT")
print("=" * 70)

print(f"Sites processed: {processed}")

print(f"Distinct sites in database: " f"{distinct_sites}")

print(f"Total weather rows: " f"{total_rows}")

print(f"Expected sites: " f"{len(data)}")

print(f"Expected rows: " f"{len(data) * 24}")

print("=" * 70)


if distinct_sites == len(data) and total_rows == len(data) * 24:

    print("SUCCESS: All sites and weather " "records were fetched and stored.")

else:

    print("WARNING: Database does not contain " "the expected final number of records.")

Total sites received from meta: 10000

PROGRESS: 1/10000
SITE: ZWA674
LATITUDE: -31.72
LONGITUDE: -36.62
Fetching: ZWA674
SUCCESS: ZWA674
Rows saved: 24/24
Progress: 1/10000
Total rows saved: 24

PROGRESS: 2/10000
SITE: FYL697
LATITUDE: -60.07
LONGITUDE: 29.57
Fetching: FYL697
SUCCESS: FYL697
Rows saved: 24/24
Progress: 2/10000
Total rows saved: 48

PROGRESS: 3/10000
SITE: KDN992
LATITUDE: 19.25
LONGITUDE: 33.91
Fetching: KDN992
SUCCESS: KDN992
Rows saved: 24/24
Progress: 3/10000
Total rows saved: 72

PROGRESS: 4/10000
SITE: NHD598
LATITUDE: 52.43
LONGITUDE: -45.11
Fetching: NHD598
SUCCESS: NHD598
Rows saved: 24/24
Progress: 4/10000
Total rows saved: 96

PROGRESS: 5/10000
SITE: YPD019
LATITUDE: -58.21
LONGITUDE: 119.42
Fetching: YPD019
SUCCESS: YPD019
Rows saved: 24/24
Progress: 5/10000
Total rows saved: 120

PROGRESS: 6/10000
SITE: BMZ568
LATITUDE: -15.46
LONGITUDE: 25.15
Fetching: BMZ568
SUCCESS: BMZ568
Rows saved: 24/24
Progress: 6/10000
Total rows saved: 144

PROGRESS: 7/10000
SITE

KeyboardInterrupt: 

In [ ]:
import time
import datetime

import psycopg
import requests
import openmeteo_requests
   

# ========================================================
# SETTINGS
# ========================================================

MAX_LOCATIONS = 7000
BATCH_SIZE = 10

START_DATE = "2026-08-11"
END_DATE = "2026-08-11"

OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"

# ------------------------------------------------------------
# Stay safely below Open-Meteo's 600 calls/minute limit.
#
# 0.20 seconds between requests = maximum theoretical
# 300 requests/minute.
# ------------------------------------------------------------

REQUEST_DELAY_SECONDS = 0.20

# ------------------------------------------------------------
# If a minute limit is nevertheless returned, wait this long.
# ------------------------------------------------------------

MINUTE_LIMIT_WAIT = 3600

# ------------------------------------------------------------
# If hourly limit is reached, wait 65 minutes.
# ------------------------------------------------------------

HOURLY_LIMIT_WAIT = 65 * 60


# ============================================================
# DATABASE SETTINGS
# ============================================================

DB_USER = "postgres"
DB_PASSWORD = "123789"
DB_HOST = "localhost"
DB_PORT = "5000"


# ============================================================
# GET REAL SITES FROM meta.metadata
# ============================================================

def get_sites_from_metadata():

    with psycopg.connect(
        dbname="meta",
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT,
    ) as conn:

        with conn.cursor() as cursor:

            cursor.execute(
                """
                SELECT
                    site_name,
                    latitude,
                    longitude
                FROM metadata
                WHERE latitude IS NOT NULL
                  AND longitude IS NOT NULL
                ORDER BY site_name
                LIMIT %s;
                """,
                (MAX_LOCATIONS,),
            )

            sites = cursor.fetchall()

    return sites


# ============================================================
# LOAD SITES
# ============================================================

data = get_sites_from_metadata()


print("=" * 70)
print("METADATA CHECK")
print("=" * 70)

print(
    f"Number of sites loaded from meta.metadata: "
    f"{len(data)}"
)


# ------------------------------------------------------------
# We require exactly 7000 sites.
# ------------------------------------------------------------

if len(data) != MAX_LOCATIONS:

    raise RuntimeError(
        f"Expected {MAX_LOCATIONS} sites in metadata, "
        f"but received {len(data)}."
    )


# ------------------------------------------------------------
# Show first 10 sites.
# This lets you verify where AAA084 came from.
# ------------------------------------------------------------

print("\nFirst 10 sites read from metadata:")

for row in data[:10]:

    print(
        f"Site={row[0]}, "
        f"Latitude={row[1]}, "
        f"Longitude={row[2]}"
    )


print("=" * 70)


# ============================================================
# OPEN-METEO CLIENT
# ============================================================

session = requests.Session()

openmeteo = openmeteo_requests.Client(
    session=session
)


# ============================================================
# WEATHER DATABASE
# ============================================================

try:

    with psycopg.connect(
        dbname="site_weather",
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT,
    ) as conn:

        with conn.cursor() as cursor:

            # =================================================
            # CREATE TABLE
            # =================================================

            cursor.execute(
                """
                CREATE TABLE IF NOT EXISTS site_weather (

                    site_name VARCHAR(100) NOT NULL,

                    time_interval TIMESTAMPTZ NOT NULL,

                    temperature REAL NOT NULL,

                    humidity REAL NOT NULL,

                    solar_radiance REAL NOT NULL,

                    UNIQUE (
                        site_name,
                        time_interval
                    )
                );
                """
            )

            conn.commit()

            print(
                "\nsite_weather table ready."
            )


            # =================================================
            # COUNTERS
            # =================================================

            locations_processed = 0

            successful_batches = 0

            successful_api_requests = 0

            failed_api_attempts = 0


            # =================================================
            # 700 BATCHES
            # =================================================

            total_batches = (
                MAX_LOCATIONS // BATCH_SIZE
            )


            for batch_start in range(
                0,
                MAX_LOCATIONS,
                BATCH_SIZE
            ):

                batch_end = (
                    batch_start + BATCH_SIZE
                )

                batch = data[
                    batch_start:batch_end
                ]


                # ------------------------------------------------
                # Safety check
                # ------------------------------------------------

                if len(batch) != BATCH_SIZE:

                    raise RuntimeError(
                        f"Batch contains "
                        f"{len(batch)} sites instead of "
                        f"{BATCH_SIZE}."
                    )


                batch_number = (
                    batch_start // BATCH_SIZE
                ) + 1


                # =================================================
                # SITE INFORMATION FROM DATABASE
                # =================================================

                site_codes = [
                    row[0]
                    for row in batch
                ]

                latitudes = [
                    float(row[1])
                    for row in batch
                ]

                longitudes = [
                    float(row[2])
                    for row in batch
                ]


                # =================================================
                # API PARAMETERS
                # =================================================

                params = {

                    "latitude": latitudes,

                    "longitude": longitudes,

                    "hourly": [
                        "temperature_2m",
                        "relative_humidity_2m",
                        "direct_radiation",
                    ],

                    "start_date": START_DATE,

                    "end_date": END_DATE,
                }


                # =================================================
                # REQUEST THE BATCH
                # =================================================

                while True:

                    try:

                        print("\n" + "=" * 70)

                        print(
                            f"BATCH "
                            f"{batch_number}/{total_batches}"
                        )

                        print(
                            f"Sites "
                            f"{batch_start + 1}-"
                            f"{batch_end}"
                        )

                        print(
                            f"Sending {len(batch)} locations "
                            f"in ONE HTTP request"
                        )

                        print(
                            f"First site: "
                            f"{site_codes[0]}"
                        )

                        print(
                            f"Last site: "
                            f"{site_codes[-1]}"
                        )

                        print("=" * 70)


                        # =================================================
                        # ONE HTTP REQUEST
                        # =================================================

                        responses = (
                            openmeteo.weather_api(
                                OPEN_METEO_URL,
                                params=params,
                                method="POST"
                            )
                        )


                        # =================================================
                        # VERIFY RESPONSE COUNT
                        # =================================================

                        if len(responses) != BATCH_SIZE:

                            raise RuntimeError(
                                f"Open-Meteo returned "
                                f"{len(responses)} responses "
                                f"instead of "
                                f"{BATCH_SIZE}."
                            )


                        # ------------------------------------------------
                        # SUCCESS
                        # ------------------------------------------------

                        successful_api_requests += 1


                        print(
                            f"\nSUCCESS!"
                        )

                        print(
                            f"One HTTP request returned "
                            f"{len(responses)} locations."
                        )

                        print(
                            f"Successful API requests: "
                            f"{successful_api_requests}"
                        )


                        break


                    except Exception as e:

                        error_message = str(e)

                        failed_api_attempts += 1


                        print(
                            "\nOpen-Meteo request failed:"
                        )

                        print(
                            error_message
                        )


                        # =================================================
                        # MINUTE LIMIT
                        # =================================================

                        if (
                            "Minutely API request limit exceeded"
                            in error_message
                        ):

                            print(
                                "\nMinute limit reached."
                            )

                            print(
                                f"Waiting "
                                f"{MINUTE_LIMIT_WAIT} "
                                f"seconds..."
                            )

                            time.sleep(
                                MINUTE_LIMIT_WAIT
                            )

                            # Same batch is retried.

                            continue


                        # =================================================
                        # HOURLY LIMIT
                        # =================================================

                        elif (
                            "Hourly API request limit exceeded"
                            in error_message
                        ):

                            print(
                                "\nHourly limit reached."
                            )

                            print(
                                "The server controls the quota."
                            )

                            print(
                                f"Waiting "
                                f"{HOURLY_LIMIT_WAIT / 60:.0f} "
                                f"minutes..."
                            )

                            time.sleep(
                                HOURLY_LIMIT_WAIT
                            )

                            # Same batch is retried.

                            continue


                        # =================================================
                        # OTHER ERROR
                        # =================================================

                        else:

                            raise


                # =================================================
                # PROCESS 10 RESPONSES
                # =================================================

                batch_locations_saved = 0


                for site_code, response in zip(
                    site_codes,
                    responses
                ):


                    # ------------------------------------------------
                    # IMPORTANT:
                    # site_code comes from metadata.
                    # We NEVER generate it.
                    # ------------------------------------------------

                    hourly = response.Hourly()


                    if hourly is None:

                        raise RuntimeError(
                            f"No hourly data returned "
                            f"for site {site_code}"
                        )


                    # =================================================
                    # WEATHER ARRAYS
                    # =================================================

                    temperatures = (
                        hourly
                        .Variables(0)
                        .ValuesAsNumpy()
                    )

                    humidity = (
                        hourly
                        .Variables(1)
                        .ValuesAsNumpy()
                    )

                    radiation = (
                        hourly
                        .Variables(2)
                        .ValuesAsNumpy()
                    )


                    # =================================================
                    # VERIFY 24 HOURS
                    # =================================================

                    number_of_hours = (
                        len(temperatures)
                    )


                    if number_of_hours != 24:

                        raise RuntimeError(
                            f"Site {site_code} "
                            f"returned "
                            f"{number_of_hours} "
                            f"hours instead of 24."
                        )


                    if (
                        len(humidity)
                        != 24
                        or
                        len(radiation)
                        != 24
                    ):

                        raise RuntimeError(
                            f"Weather array length "
                            f"problem for "
                            f"{site_code}."
                        )


                    # =================================================
                    # TIME
                    # =================================================

                    start_timestamp = (
                        hourly.Time()
                    )


                    start_time = (
                        datetime.datetime
                        .fromtimestamp(
                            start_timestamp,
                            tz=datetime.timezone.utc
                        )
                    )


                    interval_seconds = (
                        hourly.Interval()
                    )


                    # =================================================
                    # INSERT 24 HOURLY RECORDS
                    # =================================================

                    for j in range(24):

                        time_interval = (
                            start_time
                            +
                            datetime.timedelta(
                                seconds=(
                                    interval_seconds * j
                                )
                            )
                        )


                        cursor.execute(
                            """
                            INSERT INTO site_weather (
                                site_name,
                                time_interval,
                                temperature,
                                humidity,
                                solar_radiance
                            )
                            VALUES (
                                %s,
                                %s,
                                %s,
                                %s,
                                %s
                            )
                            ON CONFLICT (
                                site_name,
                                time_interval
                            )
                            DO NOTHING;
                            """,
                            (
                                site_code,

                                time_interval,

                                float(
                                    temperatures[j]
                                ),

                                float(
                                    humidity[j]
                                ),

                                float(
                                    radiation[j]
                                ),
                            )
                        )


                    batch_locations_saved += 1


                    print(
                        f"Saved: "
                        f"{site_code} "
                        f"-> 24 hourly records"
                    )


                # =================================================
                # COMMIT WHOLE BATCH
                # =================================================

                conn.commit()


                # =================================================
                # UPDATE COUNTERS
                # =================================================

                locations_processed += (
                    batch_locations_saved
                )

                successful_batches += 1


                print("\n" + "-" * 70)

                print(
                    f"BATCH {batch_number}/"
                    f"{total_batches} COMPLETE"
                )

                print(
                    f"Locations saved: "
                    f"{batch_locations_saved}/10"
                )

                print(
                    f"Total locations: "
                    f"{locations_processed}/7000"
                )

                print(
                    f"Successful batches: "
                    f"{successful_batches}/700"
                )

                print(
                    f"Successful HTTP requests: "
                    f"{successful_api_requests}"
                )

                print(
                    f"Failed/retried attempts: "
                    f"{failed_api_attempts}"
                )

                print("-" * 70)


                # =================================================
                # RATE CONTROL
                # =================================================

                # Do not immediately hammer the API again.

                time.sleep(
                    REQUEST_DELAY_SECONDS
                )


            # ====================================================
            # FINAL VERIFICATION
            # ====================================================

            print("\n")
            print("=" * 70)
            print("FINAL VERIFICATION")
            print("=" * 70)


            print(
                f"Locations processed: "
                f"{locations_processed}/7000"
            )

            print(
                f"Successful batches: "
                f"{successful_batches}/700"
            )

            print(
                f"Successful HTTP requests: "
                f"{successful_api_requests}"
            )

            print(
                f"Failed/retried attempts: "
                f"{failed_api_attempts}"
            )


            # =================================================
            # COUNT DISTINCT SITES
            # =================================================

            cursor.execute(
                """
                SELECT COUNT(DISTINCT site_name)
                FROM site_weather;
                """
            )

            distinct_sites = (
                cursor.fetchone()[0]
            )


            print(
                f"Distinct sites in "
                f"site_weather: "
                f"{distinct_sites}"
            )


            # =================================================
            # COUNT TOTAL ROWS
            # =================================================

            cursor.execute(
                """
                SELECT COUNT(*)
                FROM site_weather;
                """
            )

            total_rows = (
                cursor.fetchone()[0]
            )


            print(
                f"Total rows in "
                f"site_weather: "
                f"{total_rows}"
            )


            # =================================================
            # EXPECTED ROW COUNT
            # =================================================

            expected_rows = (
                MAX_LOCATIONS * 24
            )


            print(
                f"Expected rows: "
                f"{expected_rows}"
            )


            # =================================================
            # VERIFY MISSING SITES
            # =================================================

            cursor.execute(
                """
                SELECT COUNT(*)
                FROM metadata m
                LEFT JOIN (
                    SELECT DISTINCT site_name
                    FROM site_weather
                ) w
                    ON m.site_name = w.site_name
                WHERE w.site_name IS NULL;
                """
            )

            missing_sites = (
                cursor.fetchone()[0]
            )


            print(
                f"Metadata sites without "
                f"weather data: "
                f"{missing_sites}"
            )


            print("=" * 70)

            print("DONE.")

            print("=" * 70)


except psycopg.Error as e:

    print(
        "\nPostgreSQL error:"
    )

    print(e)


except Exception as e:

    print(
        "\nProgram error:"
    )

    print(e)

METADATA CHECK
Number of sites loaded from meta.metadata: 7000

First 10 sites read from metadata:
Site=AAB708, Latitude=83.56, Longitude=169.29
Site=AAK344, Latitude=-80.64, Longitude=-151.13
Site=AAR133, Latitude=-55.83, Longitude=153.5
Site=ABE149, Latitude=-44.25, Longitude=-132.67
Site=ABI606, Latitude=-50.4, Longitude=83.96
Site=ABN355, Latitude=-77.72, Longitude=133.94
Site=ABP466, Latitude=60.25, Longitude=66.44
Site=ABQ893, Latitude=12.31, Longitude=-107.63
Site=ABS300, Latitude=-48.73, Longitude=-158.41
Site=ABT088, Latitude=-69.18, Longitude=-174.8

site_weather table ready.

BATCH 1/700
Sites 1-10
Sending 10 locations in ONE HTTP request
First site: AAB708
Last site: ABT088

SUCCESS!
One HTTP request returned 10 locations.
Successful API requests: 1
Saved: AAB708 -> 24 hourly records
Saved: AAK344 -> 24 hourly records
Saved: AAR133 -> 24 hourly records
Saved: ABE149 -> 24 hourly records
Saved: ABI606 -> 24 hourly records
Saved: ABN355 -> 24 hourly records
Saved: ABP466 -> 2

In [ ]:
# url = "https://api.open-meteo.com/v1/forecast"

# import time

# MAX_LOCATIONS = 7000
# BATCH_SIZE = 30

# locations_processed = 0

# for i in range(0, len(data), 30):

#     batch = data[i : i + 30]

#     site_codes = [row[0] for row in batch]
#     latitudes = [row[1] for row in batch]
#     longitudes = [row[2] for row in batch]

#     params = {
#         "latitude": latitudes,
#         "longitude": longitudes,
#         "hourly": ["temperature_2m", "relative_humidity_2m", "direct_radiation"],
#     }

#     while True:
#         try:
#             responses = openmeteo.weather_api(url, params=params, method="POST")

#             print("Batch:", i, "Responses:", len(responses))
#             break  # successful request, while loop se bahar

#         except Exception as e:

#             if "Minutely API request limit exceeded" in str(e):

#                 print("API limit reached. Waiting 60 seconds...")
#                 time.sleep(60)

#             else:
#                 raise e

Batch: 0 Responses: 30
Batch: 30 Responses: 30
Batch: 60 Responses: 30
Batch: 90 Responses: 30
Batch: 120 Responses: 30
Batch: 150 Responses: 30
Batch: 180 Responses: 30
Batch: 210 Responses: 30
Batch: 240 Responses: 30
Batch: 270 Responses: 30
Batch: 300 Responses: 30
Batch: 330 Responses: 30
Batch: 360 Responses: 30
Batch: 390 Responses: 30
Batch: 420 Responses: 30
Batch: 450 Responses: 30
Batch: 480 Responses: 30
Batch: 510 Responses: 30
Batch: 540 Responses: 30
Batch: 570 Responses: 30
API limit reached. Waiting 60 seconds...
Batch: 600 Responses: 30
Batch: 630 Responses: 30
Batch: 660 Responses: 30
Batch: 690 Responses: 30
Batch: 720 Responses: 30
Batch: 750 Responses: 30
Batch: 780 Responses: 30
Batch: 810 Responses: 30
Batch: 840 Responses: 30
Batch: 870 Responses: 30
Batch: 900 Responses: 30
Batch: 930 Responses: 30
Batch: 960 Responses: 30
Batch: 990 Responses: 30
Batch: 1020 Responses: 30
Batch: 1050 Responses: 30
Batch: 1080 Responses: 30
Batch: 1110 Responses: 30
Batch: 114

KeyboardInterrupt: 

In [ ]:
# len(responses)

500

In [ ]:
# for site_code, response in zip(site_codes, responses):

#     hourly = response.Hourly()

#     temperature = hourly.Variables(0).ValuesAsNumpy()
#     humidity = hourly.Variables(1).ValuesAsNumpy()
#     radiation = hourly.Variables(2).ValuesAsNumpy()

#     # print(site_code, temperature[:5])

In [ ]:
# df = pd.DataFrame(
#         {
#             "site_code": site_code,
#             "time_interval": pd.date_range(
#                 start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
#                 end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
#                 freq=pd.Timedelta(seconds=hourly.Interval()),
#                 inclusive="right",
#             ),
#             "temperature": temperature,
#             "humidity": humidity,
#             "solar_radiance": radiation,
#         }
#     )
# df.to_csv("weather_data.csv", index=False)

In [ ]:
# def create_db_site_weather(db_name):
#     try:
#         with psycopg.connect(
#             dbname="csv_database",
#             user="postgres",
#             password="123789",
#             host="localhost",
#             port="5000",
#             autocommit=True,
#         ) as conn:
#             with conn.cursor() as cur:
#                 query = sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_name))
#                 cur.execute(query)
#                 print(f"Database '{db_name}' created successfully!")

#     except psycopg.Error as e:
#         print(f"An error occurred: {e}")

In [ ]:
# create_db_site_weather("site_weather")

Database 'site_weather' created successfully!


In [ ]:
# def create_table_site_weather():
#     try:
#         with psycopg.connect(
#             dbname="site_weather",
#             user="postgres",
#             password="123789",
#             host="localhost",
#             port="5000",
#         ) as conn:

#             with conn.cursor() as cur:
#                 cur.execute("""
#                     CREATE TABLE IF NOT EXISTS site_weather (
#                         site_name VARCHAR(100) Not NULL,
#                         time_interval TIMESTAMPTZ Not NULL,
#                         temperature REAL Not NULL,
#                         humidity REAL Not NULL,
#                         solar_radiance REAL Not NULL
#                     );
#                 """)

#             conn.commit()
#             print("site_weather table created.")

#     except psycopg.Error as e:
#         print(e)

In [ ]:
# create_table_site_weather()

site_weather table created.


In [ ]:
# def data_store(sites_df):

#     with psycopg.connect(
#         dbname="site_weather",
#         user="postgres",
#         password="123789",
#         host="localhost",
#         port="5000",
#     ) as conn:
#         with conn.cursor() as cur:

#             for _, row in sites_df.iterrows():

#                 cur.execute(
#                     """
#                     INSERT INTO site_weather
#                     (site_name, time_interval, temperature, humidity, solar_radiance)
#                     VALUES (%s, %s, %s, %s, %s)
#                     """,
#                     (
#                         row["site_code"],
#                         row["time_interval"],
#                         row["temperature"],
#                         row["humidity"],
#                         row["solar_radiance"],
#                     ),
#                 )

#         conn.commit()

#     print("Site weather data inserted successfully!")

In [ ]:
# data_store(df)

Site weather data inserted successfully!
